<a href="https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule
For Lane 2 — Refresh / Content Opportunity Scoring, I will rank pages using only information available in the current 90-day observation window.

The score combines:

1.Visibility — pages with more impressions have more potential impact.
2.Freshness risk — pages that have not been updated for a long time receive a higher review priority.
3.Position opportunity — pages with meaningful visibility and positions closer to the first page receive higher priority because they may have an opportunity for improvement.
4.Content depth gap — shorter content with meaningful visibility receives a small additional priority.
The score is a prioritization score, not a prediction that a page will recover after being refreshed.

Reason codes
The rule assigns one primary reason code:

stale_visible_page — the page is old since its last update and has meaningful impressions.
declining_with_demand — the page shows declining demand while still having meaningful impressions.
thin_visible_page — the page has relatively low word count but meaningful visibility.
low_ctr_visible_page — the page has meaningful impressions and ranking position but relatively low CTR.
page_one_decay_risk — the page is on/near page one but is relatively old.
general_refresh_review — none of the specific conditions is met.

Action labels
refresh — prioritize a content refresh.
expand_and_refresh — prioritize content expansion and refresh.
refresh_and_review_ctr — review the title/snippet/CTR opportunity.
monitor — keep the page under observation rather than immediately recommending a refresh.
The rule does not use trend_direction, trend_pct, or is_declining_label as scoring inputs because those are derived from the outcome label. The starter documentation explicitly identifies them as label-derived and warns against using them as model features. :contentReference[oaicite:3]{index=3}

In [1]:
# This cell is for CODE (numbers, a query, a check).
import os
import pandas as pd
import numpy as np

repo = "/content/flyrank-ml-internship-starter"
raw_path = os.path.join(repo, "data/raw/content_refresh_anonymized.csv")

# Clone the starter repo if it is not already available
if not os.path.exists(raw_path):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

df = pd.read_csv(raw_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# -----------------------------
# Signal 1: freshness / staleness
# -----------------------------
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda x:
                          (x.str.lower() == "down").mean())
      )
      .reset_index()
)

print("\nSIGNAL 1 — days_since_last_update")
print(staleness_check)

# -----------------------------
# Signal 2: visibility
# -----------------------------
df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 99, 499, 999, 4999, np.inf],
    labels=["1-99", "100-499", "500-999", "1k-4.9k", "5k+"]
)

visibility_check = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda x:
                          (x.str.lower() == "down").mean())
      )
      .reset_index()
)

print("\nSIGNAL 2 — impressions_90d")
print(visibility_check)


Rows: 30000
Columns: 44

SIGNAL 1 — days_since_last_update
  staleness_bucket      n  declining_rate
0             0-30  20480        0.511377
1            31-90    175        0.588571
2           91-180   9171        0.611057
3          181-365    169        0.467456
4             365+      5        0.600000

SIGNAL 2 — impressions_90d
  visibility_bucket     n  declining_rate
0              1-99  7994        0.389042
1           100-499  5280        0.604356
2           500-999  3214        0.600498
3           1k-4.9k  7361        0.634153
4               5k+  6151        0.546740


Signal checks and verdicts

Signal 1 — days_since_last_update: MIXED
The staleness signal shows some evidence that older pages may be more likely to decline. Pages updated within 0–30 days have a declining rate of 51.1%, while the 91–180 day bucket has a higher declining rate of 61.1%. However, the relationship is not consistently increasing: the 181–365 day bucket drops to 46.7%, and the 365+ bucket contains only 5 pages. Therefore, I classify this signal as MIXED and will use staleness as a supporting prioritization signal rather than a guaranteed refresh indicator.

Signal 2 — impressions_90d: MIXED
The visibility signal also shows a mixed relationship. Pages with 1–99 impressions have a declining rate of 38.9%, while the 100–499, 500–999, and 1k–4.9k buckets have higher declining rates of 60.4%, 60.0%, and 63.4%. However, the 5k+ bucket falls to 54.7%, so the relationship is not consistently increasing. Therefore, I classify this signal as MIXED and will use impressions as a supporting prioritization signal rather than assuming that higher visibility always means higher decline risk.

Combined observation
Both signals receive a MIXED verdict. Together, they provide useful evidence for a review-prioritization baseline: days_since_last_update represents content staleness, while impressions_90d represents search visibility. The results support testing these signals in a simple baseline score, while avoiding claims that either signal causes declining performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline scoring rule

The baseline score is a transparent weighted score:

40% visibility

30% freshness risk

25% position opportunity

5% content depth gap

Each component is normalized within the available dataset and combined into a score from 0 to 1.

The score is used only to rank pages for human review. It is not a prediction of future performance.

The output contains one primary reason code and one suggested action for each page.

In [2]:
# This cell is for CODE (numbers, a query, a check).
import numpy as np
import pandas as pd
from pathlib import Path

work_df = df.copy()

# Make numeric columns safe
numeric_cols = [
    "impressions_90d",
    "avg_position",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate",
    "ctr"
]

for col in numeric_cols:
    work_df[col] = pd.to_numeric(
        work_df[col], errors="coerce"
    ).fillna(0)


# -----------------------------
# Helper functions
# -----------------------------
def percentile_rank(series):
    return series.rank(method="average", pct=True).fillna(0)


def normalize(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


# -----------------------------
# Score components
# -----------------------------

# 1. Visibility
work_df["visibility_score"] = percentile_rank(
    np.log1p(work_df["impressions_90d"])
)

# 2. Freshness risk
work_df["freshness_risk_score"] = percentile_rank(
    work_df["days_since_last_update"]
)

# 3. Position opportunity
position_normalized = normalize(
    work_df["avg_position"].clip(lower=1, upper=50)
)

work_df["position_opportunity_score"] = (
    (1 - position_normalized)
    * work_df["visibility_score"]
    * (work_df["avg_position"] > 0).astype(int)
)

# 4. Content depth gap
work_df["depth_gap_score"] = (
    (1 - percentile_rank(work_df["word_count"]))
    * work_df["visibility_score"]
)


# -----------------------------
# Final baseline score
# -----------------------------

work_df["baseline_action_score"] = (
    0.40 * work_df["visibility_score"]
    + 0.30 * work_df["freshness_risk_score"]
    + 0.25 * work_df["position_opportunity_score"]
    + 0.05 * work_df["depth_gap_score"]
).clip(0, 1)


# -----------------------------
# ONE primary reason code
# -----------------------------

def get_reason(row):

    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        return "stale_visible_page"

    if (
        row["trend_direction"].lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        return "declining_with_demand"

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        return "thin_visible_page"

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        return "low_ctr_visible_page"

    if (
        0 < row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        return "page_one_decay_risk"

    return "general_refresh_review"


work_df["reason_code"] = work_df.apply(
    get_reason,
    axis=1
)


# -----------------------------
# Action label
# -----------------------------

def get_action(reason):

    if reason == "thin_visible_page":
        return "expand_and_refresh"

    if reason == "low_ctr_visible_page":
        return "refresh_and_review_ctr"

    if reason in [
        "stale_visible_page",
        "declining_with_demand",
        "page_one_decay_risk"
    ]:
        return "refresh"

    return "monitor"


work_df["action"] = work_df["reason_code"].apply(get_action)


# -----------------------------
# Rank
# -----------------------------

work_df = work_df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

work_df["baseline_rank"] = np.arange(
    1, len(work_df) + 1
)


# -----------------------------
# Create output
# -----------------------------

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update"
]

queue = work_df[output_columns].copy()

output_path = Path(
    repo
) / "work/outputs/baseline_action_score.csv"

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    output_path,
    index=False
)

print("Baseline queue created successfully.")
print("Rows:", len(queue))
print("Output:", output_path)

display(queue.head(10))
print("\nTop 10 baseline queue:")
display(queue.head(10))

print("\nAction counts:")
print(queue["action"].value_counts())

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())


Baseline queue created successfully.
Rows: 30000
Output: /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,word_count,content_age_days,days_since_last_update
0,content_9532f197bbc8,client_4e07408562,1,0.941189,declining_with_demand,refresh,309192,2689,1098,2.0,0.87,0.0,445,104
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,page_one_decay_risk,refresh,97999,512,549,2.5,0.52,0.0,329,104
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,page_one_decay_risk,refresh,101078,856,780,2.7,0.85,0.0,313,104
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,low_ctr_visible_page,refresh_and_review_ctr,117741,533,522,3.0,0.45,0.0,421,104
4,content_3430a8b94511,client_19581e27de,5,0.933559,low_ctr_visible_page,refresh_and_review_ctr,152617,440,534,3.3,0.29,0.0,329,104
5,content_cbd93118300b,client_19581e27de,6,0.933263,declining_with_demand,refresh,145292,662,535,3.3,0.46,0.0,313,104
6,content_9c195417f6ef,client_19581e27de,7,0.932991,page_one_decay_risk,refresh,79146,574,515,2.5,0.73,0.0,313,104
7,content_ba2acb4ebd04,client_19581e27de,8,0.931623,page_one_decay_risk,refresh,142072,1185,1147,3.6,0.83,0.0,362,104
8,content_79b25654070a,client_19581e27de,9,0.931363,low_ctr_visible_page,refresh_and_review_ctr,148737,711,619,3.7,0.48,0.0,257,104
9,content_adddad39251c,client_19581e27de,10,0.931124,page_one_decay_risk,refresh,129239,711,688,3.6,0.55,0.0,329,104



Top 10 baseline queue:


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,word_count,content_age_days,days_since_last_update
0,content_9532f197bbc8,client_4e07408562,1,0.941189,declining_with_demand,refresh,309192,2689,1098,2.0,0.87,0.0,445,104
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,page_one_decay_risk,refresh,97999,512,549,2.5,0.52,0.0,329,104
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,page_one_decay_risk,refresh,101078,856,780,2.7,0.85,0.0,313,104
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,low_ctr_visible_page,refresh_and_review_ctr,117741,533,522,3.0,0.45,0.0,421,104
4,content_3430a8b94511,client_19581e27de,5,0.933559,low_ctr_visible_page,refresh_and_review_ctr,152617,440,534,3.3,0.29,0.0,329,104
5,content_cbd93118300b,client_19581e27de,6,0.933263,declining_with_demand,refresh,145292,662,535,3.3,0.46,0.0,313,104
6,content_9c195417f6ef,client_19581e27de,7,0.932991,page_one_decay_risk,refresh,79146,574,515,2.5,0.73,0.0,313,104
7,content_ba2acb4ebd04,client_19581e27de,8,0.931623,page_one_decay_risk,refresh,142072,1185,1147,3.6,0.83,0.0,362,104
8,content_79b25654070a,client_19581e27de,9,0.931363,low_ctr_visible_page,refresh_and_review_ctr,148737,711,619,3.7,0.48,0.0,257,104
9,content_adddad39251c,client_19581e27de,10,0.931124,page_one_decay_risk,refresh,129239,711,688,3.6,0.55,0.0,329,104



Action counts:
action
refresh                   15976
monitor                   10350
refresh_and_review_ctr     3636
expand_and_refresh           38
Name: count, dtype: int64

Reason-code counts:
reason_code
declining_with_demand     13136
general_refresh_review    10350
low_ctr_visible_page       3636
page_one_decay_risk        2823
thin_visible_page            38
stale_visible_page           17
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The following review examines the highest-ranked pages from the baseline queue.

For each page I record:

the recommended action,

the primary reason code,

why the page was ranked highly,

and what evidence could make the recommendation wrong.

The score is a prioritization aid, not a claim that the recommended action will improve performance.

In [3]:
# This cell is for CODE (numbers, a query, a check).

top20 = queue.head(20).copy()

for _, row in top20.iterrows():

    print(
        f"Rank {row['baseline_rank']} | "
        f"Content: {row['content_id']} | "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Score: {row['baseline_action_score']:.3f}"
    )

    print(
        f"  Why: impressions={row['impressions_90d']}, "
        f"position={row['avg_position']:.1f}, "
        f"CTR={row['ctr']:.2f}%, "
        f"days since update={row['days_since_last_update']}, "
        f"word count={row['word_count']}"
    )

    print(
        "  What could make it wrong: "
        "the observed signal may not represent the page's current condition, "
        "or the underlying metric may be affected by incomplete data."
    )

    print()

Rank 1 | Content: content_9532f197bbc8 | Action: refresh | Reason: declining_with_demand | Score: 0.941
  Why: impressions=309192, position=2.0, CTR=0.87%, days since update=104, word count=0.0
  What could make it wrong: the observed signal may not represent the page's current condition, or the underlying metric may be affected by incomplete data.

Rank 2 | Content: content_4d1fe5b32dc2 | Action: refresh | Reason: page_one_decay_risk | Score: 0.935
  Why: impressions=97999, position=2.5, CTR=0.52%, days since update=104, word count=0.0
  What could make it wrong: the observed signal may not represent the page's current condition, or the underlying metric may be affected by incomplete data.

Rank 3 | Content: content_07f2e7a6f38a | Action: refresh | Reason: page_one_decay_risk | Score: 0.934
  Why: impressions=101078, position=2.7, CTR=0.85%, days since update=104, word count=0.0
  What could make it wrong: the observed signal may not represent the page's current condition, or the unde

Top-10 Review

I reviewed the top 10 pages produced by the baseline action score. The queue prioritizes pages with strong search visibility and signals suggesting that they may deserve refresh or CTR review. The recommendations are decision-support only and should be checked by a human reviewer before taking action.

Rank 1 — content_9532f197bbc8

Action: Refresh

Why: The page has very high search visibility with 309,192 impressions, an average position of 2.0, and has not been updated for 104 days. The combination of strong demand and moderate staleness makes it a high-priority review candidate.

What could make it wrong: The observed signals may not represent the page's current condition, or the underlying metrics may be affected by incomplete data.

Rank 2 — content_4d1fe5b32dc2

Action: Refresh

Why: The page has 97,999 impressions and an average position of 2.5, while its CTR is relatively low at 0.52%. It has also gone 104 days since its last update.

What could make it wrong: The observed signals may not represent the page's current condition, or the underlying metrics may be affected by incomplete data.

Rank 3 — content_07f2e7a6f38a

Action: Refresh

Why: The page has 101,078 impressions, an average position of 2.7, and has not been updated for 104 days. Its strong visibility makes it worth reviewing for possible decay or improvement opportunities.

What could make it wrong: The observed signals may not represent the page's current condition, or the underlying metrics may be affected by incomplete data.

Rank 4 — content_e5ae436f9a16

Action: Refresh and review CTR

Why: The page has 117,741 impressions and an average position of 3.0, but its CTR is only 0.45%. The combination suggests that the page receives substantial visibility but may deserve a CTR-focused review.

What could make it wrong: A low CTR may be appropriate for the queries or search context, so changing the page may not improve performance.

Rank 5 — content_3430a8b94511

Action: Refresh and review CTR

Why: The page has 152,617 impressions and an average position of 3.3, while its CTR is only 0.29%. Its high visibility and low observed CTR make it a useful candidate for human review.

What could make it wrong: The observed CTR may reflect the type of queries generating impressions, and the page may not actually need a content change.

Rank 6 — content_cbd93118300b

Action: Refresh

Why: The page has 145,292 impressions, an average position of 3.3, and has not been updated for 104 days. Its high search exposure combined with staleness makes it worth reviewing.

What could make it wrong: The page may still be performing appropriately despite being unchanged for 104 days, or the available metrics may be incomplete.

Rank 7 — content_9c195417f6ef

Action: Refresh

Why: The page has 79,146 impressions and an average position of 2.5, with 104 days since its last update. Its strong search position and visibility make it a potential review candidate.

What could make it wrong: A strong average position does not by itself mean that the content needs refreshing.

Rank 8 — content_ba2acb4ebd04

Action: Refresh

Why: The page has 142,072 impressions and an average position of 3.6, with 104 days since its last update. The baseline therefore places it among pages worth reviewing.

What could make it wrong: The page may not have a meaningful content issue, and the available search signals may not capture the full context.

Rank 9 — content_79b25654070a

Action: Refresh and review CTR

Why: The page has 148,737 impressions and an average position of 3.7, but its CTR is only 0.48%. This combination makes it a candidate for a human CTR and content review.

What could make it wrong: The CTR may be normal for the page's query mix, so a low observed CTR does not necessarily mean that a content change is required.

Rank 10 — content_adddad39251c

Action: Refresh

Why: The page has 129,239 impressions, an average position of 3.6, and 104 days since its last update. Its visibility and staleness make it a reasonable candidate for review.

What could make it wrong: The page may still be healthy, and the baseline does not prove that refreshing it will improve search performance.

Top-10 takeaway

The baseline mainly surfaces pages with substantial search visibility, relatively strong average positions, and approximately 104 days since their last update. Some pages are also flagged for CTR review because they combine high impressions with comparatively low CTR. These are prioritization signals rather than proof that a page needs a particular change. Human review is still required before taking action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline is intentionally simple, so some high-ranked pages may not actually need a refresh.

I will inspect suspicious high-ranked pages for cases where the score is driven by a single signal or where the available data may be incomplete.

The score does not use is_declining_label or trend_pct as an input. These are label-derived fields and would create leakage.

The rule also does not use any future-window information. It uses the current 90-day page-level signals available in the starter dataset.

The baseline should therefore be treated as a transparent ranking heuristic rather than a causal or predictive guarantee.




In [4]:
# This cell is for CODE (numbers, a query, a check).
print("LEAKAGE CHECK")
print("----------------")

print("Score inputs:")
print("1. visibility_score")
print("2. freshness_risk_score")
print("3. position_opportunity_score")
print("4. depth_gap_score")

print("\nLabel-derived fields excluded from score:")
print("- is_declining_label")
print("- trend_pct")
print("- trend_direction")

print("\nFuture-window inputs:")
print("- None")

print("\nTop 5 pages for additional skepticism:")

display(
    queue[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "reason_code",
            "action",
            "impressions_90d",
            "avg_position",
            "days_since_last_update",
            "word_count"
        ]
    ].head(5)
)

LEAKAGE CHECK
----------------
Score inputs:
1. visibility_score
2. freshness_risk_score
3. position_opportunity_score
4. depth_gap_score

Label-derived fields excluded from score:
- is_declining_label
- trend_pct
- trend_direction

Future-window inputs:
- None

Top 5 pages for additional skepticism:


,baseline_rank,content_id,baseline_action_score,reason_code,action,impressions_90d,avg_position,days_since_last_update,word_count
0,1,content_9532f197bbc8,0.941189,declining_with_demand,refresh,309192,2.0,104,0.0
1,2,content_4d1fe5b32dc2,0.934889,page_one_decay_risk,refresh,97999,2.5,104,0.0
2,3,content_07f2e7a6f38a,0.934080,page_one_decay_risk,refresh,101078,2.7,104,0.0
3,4,content_e5ae436f9a16,0.933606,low_ctr_visible_page,refresh_and_review_ctr,117741,3.0,104,0.0
4,5,content_3430a8b94511,0.933559,low_ctr_visible_page,refresh_and_review_ctr,152617,3.3,104,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.